# Crawling Data Detik.com


Pada tahap ini dilakukan proses *web crawling* untuk memperoleh informasi berita olahraga yang tersedia pada situs Detik Sport. Data yang dikumpulkan berupa judul berita beserta informasi tautan dari halaman berita yang ditemukan.

Proses pengambilan data dilakukan dengan memanfaatkan beberapa library Python. **Requests** digunakan untuk mengakses halaman web dan mengambil isi HTML dari situs yang dituju. Selanjutnya, **BeautifulSoup** digunakan untuk membaca struktur HTML dan menemukan elemen-elemen yang diperlukan, khususnya bagian judul serta URL berita. Setelah data berhasil diperoleh, **Pandas** digunakan untuk mengorganisasi hasil crawling ke dalam bentuk DataFrame sehingga data dapat ditampilkan dan diolah dengan lebih mudah.

Dengan tahapan tersebut, informasi dari halaman Detik Sport dapat dikumpulkan secara otomatis dan disusun menjadi dataset yang lebih terstruktur untuk keperluan analisis selanjutnya.


Cell 1 — Import library

In [16]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

Cell 2 — Pengaturan crawling

In [17]:
BASE_URL = "https://sport.detik.com"
URL_INDEKS = "https://sport.detik.com/indeks"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/131.0.0.0 Safari/537.36"
    )
}

TARGET_DATA = 200

session = requests.Session()
session.headers.update(HEADERS)

print("Target data:", TARGET_DATA)

Target data: 200


Cell 3 — Fungsi mengambil URL berita

In [18]:
def ambil_url_berita(url_halaman):
    try:
        response = session.get(url_halaman, timeout=15)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        daftar_url = []

        for link in soup.find_all("a", href=True):
            href = link.get("href")

            if not href:
                continue

            # Mengambil link artikel Detik Sport
            if href.startswith("/"):
                href = BASE_URL + href

            if (
                "sport.detik.com" in href
                and href not in daftar_url
                and href.startswith("https://")
            ):
                daftar_url.append(href)

        return daftar_url

    except Exception as error:
        print("Gagal mengambil halaman:", error)
        return []

Cell 4 — Mengambil URL dari beberapa halaman indeks

In [19]:
semua_url = []

for halaman in range(1, 15):

    if halaman == 1:
        url = URL_INDEKS
    else:
        url = f"{URL_INDEKS}/{halaman}"

    print(f"Mengambil halaman indeks {halaman}...")

    url_ditemukan = ambil_url_berita(url)

    for item in url_ditemukan:
        if item not in semua_url:
            semua_url.append(item)

    print("Jumlah URL terkumpul:", len(semua_url))

    time.sleep(1)

    if len(semua_url) >= TARGET_DATA * 2:
        break

print("\nTotal URL yang terkumpul:", len(semua_url))

Mengambil halaman indeks 1...
Jumlah URL terkumpul: 44
Mengambil halaman indeks 2...
Gagal mengambil halaman: 404 Client Error: Not Found for url: https://sport.detik.com/indeks/2
Jumlah URL terkumpul: 44
Mengambil halaman indeks 3...
Gagal mengambil halaman: 404 Client Error: Not Found for url: https://sport.detik.com/indeks/3
Jumlah URL terkumpul: 44
Mengambil halaman indeks 4...
Gagal mengambil halaman: 404 Client Error: Not Found for url: https://sport.detik.com/indeks/4
Jumlah URL terkumpul: 44
Mengambil halaman indeks 5...
Gagal mengambil halaman: 404 Client Error: Not Found for url: https://sport.detik.com/indeks/5
Jumlah URL terkumpul: 44
Mengambil halaman indeks 6...
Gagal mengambil halaman: 404 Client Error: Not Found for url: https://sport.detik.com/indeks/6
Jumlah URL terkumpul: 44
Mengambil halaman indeks 7...
Gagal mengambil halaman: 404 Client Error: Not Found for url: https://sport.detik.com/indeks/7
Jumlah URL terkumpul: 44
Mengambil halaman indeks 8...
Gagal mengambil

Cell 5 — Fungsi mengambil isi berita

In [21]:
def ambil_isi_berita(url):
    try:
        response = session.get(url, timeout=15)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        # Beberapa kemungkinan class untuk isi artikel
        pola_konten = [
            "detail__body-text",
            "detail__body",
            "itp_bodycontent"
        ]

        konten = None

        for pola in pola_konten:
            konten = soup.find(
                "div",
                class_=lambda value: value and pola in value
            )

            if konten:
                break

        if konten is None:
            return None

        paragraf = konten.find_all("p")

        teks = []

        for p in paragraf:
            isi_p = p.get_text(" ", strip=True)

            if isi_p:
                teks.append(isi_p)

        hasil = " ".join(teks)

        # Membersihkan spasi berlebih
        hasil = re.sub(r"\s+", " ", hasil).strip()

        return hasil if hasil else None

    except Exception as error:
        return None

Cell 6 — Crawling isi berita sampai 200 data

In [22]:
data_berita = []

for nomor, url in enumerate(semua_url, start=1):

    if len(data_berita) >= TARGET_DATA:
        break

    isi = ambil_isi_berita(url)

    if isi:
        data_berita.append({
            "id": len(data_berita) + 1,
            "isi berita": isi,
            "label": "olahraga"
        })

        print(
            f"Data {len(data_berita)}/{TARGET_DATA} berhasil diambil"
        )
    else:
        print(f"Artikel ke-{nomor} tidak memiliki isi yang ditemukan")

    time.sleep(1)

print("\n==============================")
print("Crawling selesai")
print("Total data:", len(data_berita))
print("==============================")

Artikel ke-1 tidak memiliki isi yang ditemukan
Artikel ke-2 tidak memiliki isi yang ditemukan
Artikel ke-3 tidak memiliki isi yang ditemukan
Artikel ke-4 tidak memiliki isi yang ditemukan
Artikel ke-5 tidak memiliki isi yang ditemukan
Artikel ke-6 tidak memiliki isi yang ditemukan
Artikel ke-7 tidak memiliki isi yang ditemukan
Artikel ke-8 tidak memiliki isi yang ditemukan
Artikel ke-9 tidak memiliki isi yang ditemukan
Artikel ke-10 tidak memiliki isi yang ditemukan
Artikel ke-11 tidak memiliki isi yang ditemukan
Artikel ke-12 tidak memiliki isi yang ditemukan
Artikel ke-13 tidak memiliki isi yang ditemukan
Artikel ke-14 tidak memiliki isi yang ditemukan
Artikel ke-15 tidak memiliki isi yang ditemukan
Artikel ke-16 tidak memiliki isi yang ditemukan
Artikel ke-17 tidak memiliki isi yang ditemukan
Artikel ke-18 tidak memiliki isi yang ditemukan
Data 1/200 berhasil diambil
Data 2/200 berhasil diambil
Data 3/200 berhasil diambil
Data 4/200 berhasil diambil
Data 5/200 berhasil diambil
Data 

Cell 7 — Membuat DataFrame

In [25]:
df_berita = pd.DataFrame(
    data_berita,
    columns=["id", "isi berita", "label"]
)

df_berita.head(10).style.hide(axis="index")

id,isi berita,label
1,"Asian Games 2026 tak lama lagi akan dimulai. Ketua Umum Pengurus Besar Taekwondo Indonesia (PBTI) , Letjen TNI Richard Tampubolon, menyampaikan optimismennya terkait cabang olahraga yang dia pimpin. Asian Games 2026 berlangsung di Aichi-Nagoya, Jepang. Ajang multi-event olahraga Asia itu akan resmi dibuka pada 19 September dan ditutup pada 4 Oktober 2026. Presiden RI Prabowo Subianto secara resmi melepas kontingen Indonesia di Istana Kepresidenan, Jakarta Pusat, Rabu (9/9/2026). Ketua Umum Pengurus Besar Taekwondo Indonesia (PBTI) Letjen TNI Richard Tampubolon turut hadir dalam acara tersebut. SCROLL TO CONTINUE WITH CONTENT Dalam kesempatan tersebut, Prabowo berpesan kepada para atlet untuk menjaga konsistensi latihan. Orang nomor satu di Indonesia itu juga berharap para perwakilan Indonesia bisa memperlihatkan karakter bangsa. ""Kunci utama untuk meraih prestasi tertinggi adalah konsistensi dalam berlatih dan semangat pantang menyerah. Ia berharap para atlet dapat menunjukkan karakter bangsa Indonesia yang tangguh saat bertanding di Jepang nanti,"" kata PRabowo. Pesan tersebut menjadi motivasi bagi para atlet Indonesia, termasuk para wakil Taekwondo Indonesia yang akan membawa Merah Putih di Aichi-Nagoya. ""Dengan persiapan dan evaluasi yang berkesinambungan, saya yakin para atlet Taekwondo dapat memberikan kontribusi maksimal untuk kejayaan Indonesia,"" kata Richard Tampubolon. Asian Games Aichi-Nagoya 2026 akan menjadi salah satu panggung penting bagi Taekwondo Indonesia. Hasil baik nantinya bisa menjadibukti keberhasilanPBTI dalam pembinaan dan persiapan yang telah dilakukan sebelum menghadapi persaingan di level Asia.",olahraga
2,"PT Bank Negara Indonesia (Persero) Tbk atau BNI kembali memberikan dukungan terhadap perjuangan bulutangkis Indonesia di panggung internasional. Hal ini seiring ditetapkannya 20 pebulu tangkis yang akan memperkuat Merah Putih pada Asian Games 2026 Aichi-Nagoya, Jepang. Dari 20 pemain yang dipilih Persatuan Bulutangkis Seluruh Indonesia (PP PBSI), sebanyak 13 pemain akan menjalani debut di Asian Games. Mereka antara lain Alwi Farhan, Moh Zaki Ubaidillah, Muhammad Shohibul Fikri, Nikolaus Joaquin, Mutiara Ayu Puspitasari, Rachel Allessya Rose, hingga Nita Violina Marwah. Sementara itu, tujuh pemain lainnya telah memiliki pengalaman tampil di Asian Games, yakni Jonatan Christie, Fajar Alfian, Leo Rolly Carnando, Daniel Marthin, Putri Kusuma Wardani, Febriana Dwipuji Kusuma, dan Siti Fadia Silva Ramadhanti. SCROLL TO CONTINUE WITH CONTENT Corporate Secretary BNI Okki Rushartomo mengatakan komposisi pemain berpengalaman dan debutan menunjukkan proses regenerasi yang terus berlangsung dalam bulutangkis Indonesia. Asian Games Aichi-Nagoya menjadi kesempatan bagi para pemain untuk menguji kemampuan sekaligus menambah pengalaman di level internasional. ""BNI berharap seluruh atlet yang memperkuat Indonesia dapat memberikan kemampuan terbaiknya. Kehadiran para pemain berpengalaman sekaligus debutan menunjukkan bahwa regenerasi terus berjalan dan menjadi bagian penting dalam menjaga kesinambungan prestasi bulutangkis Indonesia,"" ujar Okki dalam keterangan tertulis, Rabu (9/9/2026). Asian Games 2026 juga menghadirkan kombinasi baru di sektor ganda campuran. Nikolaus Joaquin dan Siti Fadia Silva Ramadhanti dijadwalkan tampil sebagai pasangan pada ajang tersebut setelah sebelumnya masing-masing berkompetisi dengan pasangan berbeda. Menurut Okki, hadirnya kombinasi pemain baru memberikan kesempatan untuk memperluas pengalaman bertanding sekaligus menambah alternatif bagi Indonesia dalam menghadapi persaingan di level Asia. ""Yang paling utama, kami berharap tim Indonesia dapat tampil maksimal pada nomor beregu, baik putra maupun putri. Setelah itu, kami juga berharap para atlet mampu menunjukkan performa terbaik pada nomor perorangan,"" kata Okki. Nomor beregu bulutangkis Asian Games 2026 dijadwalkan berlangsung pada 20-24 September 2026, kemudian dilanjutkan dengan nomor perorangan pa

Cell 8 — Mengecek jumlah dan kolom

In [26]:
print("Jumlah baris :", len(df_berita))
print("Jumlah kolom :", len(df_berita.columns))
print("Nama kolom   :", list(df_berita.columns))

Jumlah baris : 20
Jumlah kolom : 3
Nama kolom   : ['id', 'isi berita', 'label']


Cell 9 — Simpan menjadi Excel

In [28]:
nama_file = "hasil_crawling_detiksport.xlsx"

df_berita.to_excel(
    nama_file,
    index=False,
    engine="openpyxl"
)

print("File Excel berhasil dibuat:")
print(nama_file)

File Excel berhasil dibuat:
hasil_crawling_detiksport.xlsx


Cell 10 — Tampilkan hasil akhir

In [30]:
df_berita.head(10).style.hide(axis="index")

id,isi berita,label
1,"Asian Games 2026 tak lama lagi akan dimulai. Ketua Umum Pengurus Besar Taekwondo Indonesia (PBTI) , Letjen TNI Richard Tampubolon, menyampaikan optimismennya terkait cabang olahraga yang dia pimpin. Asian Games 2026 berlangsung di Aichi-Nagoya, Jepang. Ajang multi-event olahraga Asia itu akan resmi dibuka pada 19 September dan ditutup pada 4 Oktober 2026. Presiden RI Prabowo Subianto secara resmi melepas kontingen Indonesia di Istana Kepresidenan, Jakarta Pusat, Rabu (9/9/2026). Ketua Umum Pengurus Besar Taekwondo Indonesia (PBTI) Letjen TNI Richard Tampubolon turut hadir dalam acara tersebut. SCROLL TO CONTINUE WITH CONTENT Dalam kesempatan tersebut, Prabowo berpesan kepada para atlet untuk menjaga konsistensi latihan. Orang nomor satu di Indonesia itu juga berharap para perwakilan Indonesia bisa memperlihatkan karakter bangsa. ""Kunci utama untuk meraih prestasi tertinggi adalah konsistensi dalam berlatih dan semangat pantang menyerah. Ia berharap para atlet dapat menunjukkan karakter bangsa Indonesia yang tangguh saat bertanding di Jepang nanti,"" kata PRabowo. Pesan tersebut menjadi motivasi bagi para atlet Indonesia, termasuk para wakil Taekwondo Indonesia yang akan membawa Merah Putih di Aichi-Nagoya. ""Dengan persiapan dan evaluasi yang berkesinambungan, saya yakin para atlet Taekwondo dapat memberikan kontribusi maksimal untuk kejayaan Indonesia,"" kata Richard Tampubolon. Asian Games Aichi-Nagoya 2026 akan menjadi salah satu panggung penting bagi Taekwondo Indonesia. Hasil baik nantinya bisa menjadibukti keberhasilanPBTI dalam pembinaan dan persiapan yang telah dilakukan sebelum menghadapi persaingan di level Asia.",olahraga
2,"PT Bank Negara Indonesia (Persero) Tbk atau BNI kembali memberikan dukungan terhadap perjuangan bulutangkis Indonesia di panggung internasional. Hal ini seiring ditetapkannya 20 pebulu tangkis yang akan memperkuat Merah Putih pada Asian Games 2026 Aichi-Nagoya, Jepang. Dari 20 pemain yang dipilih Persatuan Bulutangkis Seluruh Indonesia (PP PBSI), sebanyak 13 pemain akan menjalani debut di Asian Games. Mereka antara lain Alwi Farhan, Moh Zaki Ubaidillah, Muhammad Shohibul Fikri, Nikolaus Joaquin, Mutiara Ayu Puspitasari, Rachel Allessya Rose, hingga Nita Violina Marwah. Sementara itu, tujuh pemain lainnya telah memiliki pengalaman tampil di Asian Games, yakni Jonatan Christie, Fajar Alfian, Leo Rolly Carnando, Daniel Marthin, Putri Kusuma Wardani, Febriana Dwipuji Kusuma, dan Siti Fadia Silva Ramadhanti. SCROLL TO CONTINUE WITH CONTENT Corporate Secretary BNI Okki Rushartomo mengatakan komposisi pemain berpengalaman dan debutan menunjukkan proses regenerasi yang terus berlangsung dalam bulutangkis Indonesia. Asian Games Aichi-Nagoya menjadi kesempatan bagi para pemain untuk menguji kemampuan sekaligus menambah pengalaman di level internasional. ""BNI berharap seluruh atlet yang memperkuat Indonesia dapat memberikan kemampuan terbaiknya. Kehadiran para pemain berpengalaman sekaligus debutan menunjukkan bahwa regenerasi terus berjalan dan menjadi bagian penting dalam menjaga kesinambungan prestasi bulutangkis Indonesia,"" ujar Okki dalam keterangan tertulis, Rabu (9/9/2026). Asian Games 2026 juga menghadirkan kombinasi baru di sektor ganda campuran. Nikolaus Joaquin dan Siti Fadia Silva Ramadhanti dijadwalkan tampil sebagai pasangan pada ajang tersebut setelah sebelumnya masing-masing berkompetisi dengan pasangan berbeda. Menurut Okki, hadirnya kombinasi pemain baru memberikan kesempatan untuk memperluas pengalaman bertanding sekaligus menambah alternatif bagi Indonesia dalam menghadapi persaingan di level Asia. ""Yang paling utama, kami berharap tim Indonesia dapat tampil maksimal pada nomor beregu, baik putra maupun putri. Setelah itu, kami juga berharap para atlet mampu menunjukkan performa terbaik pada nomor perorangan,"" kata Okki. Nomor beregu bulutangkis Asian Games 2026 dijadwalkan berlangsung pada 20-24 September 2026, kemudian dilanjutkan dengan nomor perorangan pa

Hasil crawling yang diperoleh dari halaman Detik Sport kemudian ditampilkan dalam bentuk tabel menggunakan DataFrame Pandas. Tabel tersebut berisi kumpulan judul berita olahraga beserta URL dari masing-masing berita yang berhasil ditemukan. Data ini merupakan data awal yang selanjutnya dapat digunakan untuk proses pengolahan dan pembersihan teks sebelum dilakukan analisis pada tahap berikutnya.
